In [0]:
#| default_exp write

## Writing and changing

Cell creation, targeted updates, documentation insertion, and examples.

Writing notebooks safely is harder than appending text to a file. A notebook edit needs to preserve cell ids, clear stale outputs, validate Python when possible, optionally export through nbdev, and avoid overwriting the wrong cell.

This notebook provides the public write path: append or insert cells with `write_nb`, surgically change one cell with `update_cell`, and keep the older TOML operation workflow available through `apply_nb`.

In [ ]:
#| export
import glob
import tomllib
from pathlib import Path

from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import read_nb as _read_nb
from fastcore.nbio import write_nb as _write_nb
from fastcore.script import Param, call_parse
from nbdev.doclinks import nbdev_export

from nbskill.execute import run_notebook_test
from nbskill.review import run_style_check
from nbskill.foundation import (
    cell_hash, cell_matches_hash, cell_source, clear_outputs, cli_error,
    cli_return, find_cell_by_id, find_cell_by_text, load_cells_text,
    one_chapter, parse_cells, parse_one_cell, replace_cell,
    stamp_notebook_metadata, tracked_call, validate_code_cells,
)
from nbskill.parallel import notebook_locks

### Adding cells

`write_nb` is the broad insertion tool. It can append, insert before or after a stable cell id, replace a whole notebook, or write into a named chapter while preserving notebook structure.

In [ ]:
#| export
def _literal_replacement_mode(old_str, new_str):
    return old_str is not None or new_str is not None


def _resolve_notebook_paths(path):
    raw = str(path)
    pth = Path(raw).expanduser()
    if any(char in raw for char in "*?[]"):
        candidates = [Path(item) for item in glob.glob(raw, recursive=True)]
    elif pth.is_dir():
        candidates = list(pth.rglob("*.ipynb"))
    elif pth.is_file():
        candidates = [pth]
    else:
        candidates = []
    paths = sorted({candidate for candidate in candidates if candidate.suffix == ".ipynb" and ".ipynb_checkpoints" not in candidate.parts})
    if not paths: cli_error(f"No notebooks matched {path!r}")
    return paths


def _replace_literal_in_notebook(nb, old_str, new_str, validate_code=True):
    changed_cells, matches = 0, 0
    for cell in nb.cells:
        before = cell_source(cell)
        count = before.count(old_str)
        if not count: continue
        after = before.replace(old_str, new_str)
        if validate_code and getattr(cell, "cell_type", None) == "code": validate_code_cells([mk_cell(after)])
        cell.source = after
        clear_outputs(cell)
        changed_cells += 1
        matches += count
    if matches: stamp_notebook_metadata(nb)
    return changed_cells, matches


def _write_literal_replacements(path, old_str, new_str, export=True, run_test=False, validate_code=True, dry_run=False):
    if old_str in {None, ""}: cli_error("Pass a non-empty old_str for literal replacements")
    if new_str is None: cli_error("Pass new_str for literal replacements")
    paths = _resolve_notebook_paths(path)
    changed = []
    with notebook_locks(*paths):
        for nb_path in paths:
            nb = _read_nb(nb_path)
            cells_changed, matches = _replace_literal_in_notebook(nb, old_str, new_str, validate_code=validate_code)
            if not matches: continue
            changed.append((nb_path, cells_changed, matches))
            if not dry_run:
                _write_nb(nb, nb_path)
                if export: nbdev_export(path=str(nb_path))
                if run_test: run_notebook_test(nb_path)
    total_matches = sum(matches for _, _, matches in changed)
    total_cells = sum(cells for _, cells, _ in changed)
    if not changed:
        msg = f"No matches for {old_str!r} in {len(paths)} notebook(s)"
    else:
        prefix = "Dry run: would replace" if dry_run else "Replaced"
        msg = f"{prefix} {total_matches} matches in {total_cells} cells across {len(changed)} notebook(s)"
        details = "; ".join(f"{nb_path}: {matches} matches/{cells} cells" for nb_path, cells, matches in changed)
        msg += f" ({details})"
        if export and not dry_run: msg += " and exported with nbdev"
    print(msg)
    return cli_return([path for path, _, _ in changed])


@call_parse
@tracked_call
def write_nb(
    path: str,  # Notebook path, directory, or glob when replacing literals
    cells: Param("Cell block text", str, opt=False, nargs="?") = "",  # Cells to write; use - to read stdin
    cells_file: str | None = None,  # Read cell block text from a UTF-8 file to avoid shell escaping
    before_id: str | None = None,  # Insert before this stable cell id
    after_id: str | None = None,  # Insert after this stable cell id
    chapter: str | None = None,  # Chapter title string or regex; missing chapters are created
    replace: bool = False,  # Replace the full notebook, or the selected chapter body
    cell_type: str = "code",  # Default type for cells without %% marker
    export: bool = True,  # Run nbdev-export after writing
    run_test: bool = False,  # Execute the notebook with execnb after writing
    run_style: bool = False,  # Run style_check after writing
    style_strict: bool = False,  # Fail when style_check finds hints
    validate_code: bool = True,  # Validate new Python code cells before writing
    old_str: str | None = None,  # Literal text to replace across notebook cell sources
    new_str: str | None = None,  # Literal replacement text for old_str
    dry_run: bool = False,  # Show literal replacement plan without writing
):
    "Write cells to a notebook, or replace literal text across notebooks."
    if _literal_replacement_mode(old_str, new_str):
        if cells or cells_file or before_id or after_id or chapter or replace:
            cli_error("Literal replacement mode only accepts path, old_str, new_str, export, run_test, validate_code, and dry_run")
        return _write_literal_replacements(path, old_str, new_str, export=export, run_test=run_test, validate_code=validate_code, dry_run=dry_run)
    if dry_run: cli_error("dry_run is only supported with old_str/new_str literal replacement mode")
    if before_id and after_id: cli_error("Use either before_id or after_id, not both")
    if (before_id or after_id) and chapter is not None: cli_error("Use id-based insertion or chapter insertion, not both")
    if (before_id or after_id) and replace: cli_error("Use id-based insertion or replace, not both")
    path = Path(path)
    cells = load_cells_text(cells, cells_file)
    new_cells = parse_cells(cells, cell_type)
    if validate_code: validate_code_cells(new_cells)

    with notebook_locks(path):
        if replace and chapter is None:
            nb = new_nb(new_cells)
        else:
            nb = _read_nb(path) if path.exists() else new_nb([])
            if chapter is not None:
                span = one_chapter(nb.cells, chapter, create=True)
                if replace:
                    del nb.cells[span["start"] + 1:span["end"]]
                    target = span["start"] + 1
                else:
                    target = span["end"]
            elif before_id or after_id:
                idx, _ = find_cell_by_id(nb.cells, before_id or after_id)
                target = idx if before_id else idx + 1
            else:
                target = len(nb.cells)
            for offset, cell in enumerate(new_cells):
                nb.cells.insert(target + offset, cell)

        stamp_notebook_metadata(nb)
        _write_nb(nb, path)
        if export: nbdev_export(path=str(path))
        msg = f"Wrote {len(nb.cells)} cells to {path}"
        if replace: msg += " using replace"
        if chapter is not None: msg += f" in chapter {chapter!r}"
        if before_id: msg += f" before id={before_id}"
        if after_id: msg += f" after id={after_id}"
        if export: msg += " and exported with nbdev"
        print(msg)
        if run_test: run_notebook_test(path)
        if run_style:
            print(f"Running style_check on {path}")
            run_style_check(path, strict=style_strict)
    return cli_return(path)

In [ ]:
import tempfile as _tempfile
from pathlib import Path as _Path
from fastcore.nbio import read_nb as _read_nb
from nbskill.write import update_cell, write_nb

with _tempfile.TemporaryDirectory() as td:
    path = _Path(td) / "write.ipynb"
    write_nb(str(path), "%%code\nvalue = 1\nvalue = value + 1", replace=True, export=False)
    cell = _read_nb(path).cells[0]
    assert cell.metadata["nbskill"]["cell_type"] == "code"
    update_cell(str(path), "value = 2", cell_id=cell.id, line_range="2", export=False)
    assert _read_nb(path).cells[0].source == "value = 1\nvalue = 2"
    assert "source_hash" in _read_nb(path).cells[0].metadata["nbskill"]
    update_cell(str(path), "", cell_id=cell.id, line_range="1", export=False)
    assert _read_nb(path).cells[0].source == "value = 2"

In [0]:
import tempfile as _tempfile
from contextlib import redirect_stdout as _redirect_stdout
from io import StringIO as _StringIO
from pathlib import Path as _Path
from fastcore.nbio import read_nb as _read_nb
from nbskill.write import write_nb

with _tempfile.TemporaryDirectory() as td:
    root = _Path(td)
    one = root / "one.ipynb"
    two = root / "two.ipynb"
    write_nb(str(one), "%%code\ndef old_name():\n    return 1\n---\n%%markdown\nold_name docs", replace=True, export=False)
    write_nb(str(two), "%%code\nvalue = old_name()", replace=True, export=False)
    out = _StringIO()
    with _redirect_stdout(out):
        write_nb(str(root), old_str="old_name", new_str="new_name", dry_run=True, export=False)
    assert "Dry run: would replace" in out.getvalue()
    assert "old_name" in _read_nb(one).cells[0].source
    write_nb(str(root), old_str="old_name", new_str="new_name", export=False)
    assert "new_name" in _read_nb(one).cells[0].source
    assert "new_name docs" in _read_nb(one).cells[1].source
    assert "new_name" in _read_nb(two).cells[0].source
    assert _read_nb(two).cells[0].metadata["nbskill"]["cell_type"] == "code"

In [ ]:
#| export
def _save_nb(nb, path, export=True):
    with notebook_locks(path):
        stamp_notebook_metadata(nb)
        _write_nb(nb, path)
        if export: nbdev_export(path=str(path))

### Updating one cell

`update_cell` is the surgical tool. It keeps the original cell id, can replace a whole cell or only a line range, clears stale outputs, and can require a source hash so stale edits fail loudly.

In [ ]:
#| export
def _parse_line_range(line_range, n_lines):
    if line_range is None: return None
    value = str(line_range).strip()
    if not value: return None
    if ":" in value:
        start_s, end_s = value.split(":", 1)
        start = int(start_s) if start_s else 1
        end = int(end_s) if end_s else n_lines
    else:
        start = end = int(value)
    if start < 1 or end < start or end > n_lines:
        cli_error(f"line_range must be 1-based and within 1:{n_lines}; got {line_range!r}")
    return start - 1, end


def _replace_line_range(source, line_range, new):
    lines = source.splitlines()
    start, end = _parse_line_range(line_range, len(lines) or 1)
    replacement = [] if new == "" else new.splitlines()
    return "\n".join([*lines[:start], *replacement, *lines[end:]])


@call_parse
@tracked_call
def update_cell(
    path: str,  # Notebook path
    new: Param("Replacement cell source, replacement text, or line-range replacement", str, opt=False, nargs="?") = "",
    new_file: str | None = None,  # Read replacement text from a UTF-8 file
    cell_id: str | None = None,  # Stable notebook cell id to update
    old_str: str | None = None,  # Text to replace, or text used to find the target cell
    line_range: str | None = None,  # 1-based inclusive lines to replace, e.g. 3 or 3:5
    source_hash: str | None = None,  # Expected current source SHA256 prefix
    cell_type: str = "code",  # Default type for whole-cell replacements without %% marker
    export: bool = True,  # Run nbdev-export after writing
    run_test: bool = False,  # Execute the notebook with execnb after writing
    validate_code: bool = True,  # Validate changed Python code before writing
    dry_run: bool = False,  # Show the update plan without writing
):
    "Update one notebook cell by id, replace old_str, or replace a 1-based line range."
    if cell_id is None and old_str is None: cli_error("Pass --cell_id, --old_str, or both")
    if line_range is not None and cell_id is None: cli_error("Pass --cell_id with --line_range")
    path = Path(path)
    new = load_cells_text(new, new_file)

    with notebook_locks(path):
        nb = _read_nb(path)
        idx, cell = find_cell_by_id(nb.cells, cell_id) if cell_id else find_cell_by_text(nb.cells, old_str)
        if old_str is not None and old_str not in cell_source(cell):
            cli_error(f"old_str was not found in id={cell.id}")
        if not cell_matches_hash(cell, source_hash):
            actual = cell_hash(cell, n=None)
            cli_error(f"Hash mismatch for id={cell.id}: expected {source_hash}, actual {actual[:12]}")

        before_hash = cell_hash(cell)
        if line_range is not None:
            replacement = _replace_line_range(cell_source(cell), line_range, new)
            if validate_code and getattr(cell, "cell_type", None) == "code": validate_code_cells([mk_cell(replacement)])
            after_hash = cell_hash(replacement)
            mode = f"lines {line_range}"
            if not dry_run:
                cell.source = replacement
                clear_outputs(cell)
        elif old_str is None:
            new_cell = parse_one_cell(new, cell_type)
            if validate_code: validate_code_cells([new_cell])
            clear_outputs(new_cell)
            if not dry_run: replace_cell(nb, idx, new_cell)
            after_hash = cell_hash(new_cell)
            mode = "cell"
        else:
            replacement = cell_source(cell).replace(old_str, new, 1)
            if validate_code and getattr(cell, "cell_type", None) == "code": validate_code_cells([mk_cell(replacement)])
            after_hash = cell_hash(replacement)
            mode = "text"
            if not dry_run:
                cell.source = replacement
                clear_outputs(cell)

        msg = f"{'Dry run: would update' if dry_run else 'Updated'} {mode} id={cell.id} hash={before_hash}->{after_hash}"
        if dry_run:
            print(msg)
            return cli_return(path)
        stamp_notebook_metadata(nb)
        _write_nb(nb, path)
        if export: nbdev_export(path=str(path))
        if export: msg += " and exported with nbdev"
        print(msg)
        if run_test: run_notebook_test(path)
    return cli_return(path)

### Applying saved operations

The TOML workflow is kept for older or batch-style integrations. It lets a caller describe a `write_nb` or `update_cell` operation in a sidecar file, execute it, and then clean up temporary inputs.

In [0]:
#| export
def _read_apply_nb_spec(spec_path):
    if spec_path == "-":
        import sys
        raw = sys.stdin.read()
        base = Path.cwd()
    else:
        path = Path(spec_path).expanduser()
        raw = path.read_text(encoding="utf-8")
        base = path.parent
    spec = tomllib.loads(raw)
    params = dict(spec.get("params", {}))
    for key, value in spec.items():
        if key != "params": params.setdefault(key, value)
    tool = params.pop("tool", params.pop("action", "write_nb"))
    cleanup = params.pop("cleanup", True)
    return tool, params, cleanup, base

In [0]:
#| export
def _resolve_sidecar_path(value, base):
    if value is None: return None
    path = Path(value).expanduser()
    return path if path.is_absolute() else base / path

In [0]:
#| export
def _remove_if_sidecar(path, base):
    path = Path(path).expanduser()
    try:
        resolved, root = path.resolve(), Path(base).resolve()
        resolved.relative_to(root)
    except (OSError, ValueError):
        return False
    if resolved.exists() and resolved.is_file():
        resolved.unlink()
        return True
    return False

In [0]:
#| export
def _cleanup_apply_nb_inputs(spec_path, params, base, cleanup):
    if not cleanup or spec_path == "-": return []
    removed = []
    for key in ("cells_file", "new_file"):
        if params.get(key) and _remove_if_sidecar(_resolve_sidecar_path(params[key], base), base):
            removed.append(str(_resolve_sidecar_path(params[key], base)))
    if _remove_if_sidecar(spec_path, base): removed.append(str(Path(spec_path).expanduser()))
    return removed

In [ ]:
#| export
@call_parse
@tracked_call
def apply_nb(
    spec_path: Param("TOML operation file; use - to read TOML from stdin", str, opt=False, nargs="?") = "dev/nbskill-op.toml",
):
    "Apply a notebook operation from a TOML file and clean up dev sidecar files."
    tool, params, cleanup, base = _read_apply_nb_spec(spec_path)
    cleanup_params = dict(params)
    for key in ("cells_file", "new_file"):
        if params.get(key): params[key] = str(_resolve_sidecar_path(params[key], base))
    tools = {
        "write_nb": write_nb,
        "update_cell": update_cell,
    }
    if tool not in tools: cli_error(f"Unknown apply_nb tool {tool!r}; expected one of {', '.join(tools)}")
    result = tools[tool](**params)
    removed = _cleanup_apply_nb_inputs(spec_path, cleanup_params, base, cleanup)
    if removed: print("Removed " + ", ".join(removed))
    return cli_return(result)

In [0]:
import tempfile as _tempfile
from pathlib import Path as _Path

from fastcore.nbio import read_nb as _read_nb
from nbskill.write import apply_nb

with _tempfile.TemporaryDirectory() as td:
    root = _Path(td)
    dev = root / "dev"
    dev.mkdir()
    spec = dev / "nbskill-op.toml"
    spec.write_text(
        "\n".join([
            'tool = "write_nb"',
            'path = "demo.ipynb"',
            "replace = true",
            "export = false",
            'cells = """',
            "%%markdown",
            "## Scratch",
            "---",
            "%%code",
            "value = 3",
            '"""',
        ]),
        encoding="utf-8",
    )
    old_cwd = _Path.cwd()
    try:
        import os as _os
        _os.chdir(root)
        apply_nb(str(spec))
    finally:
        _os.chdir(old_cwd)
    assert not spec.exists()
    nb = _read_nb(root / "demo.ipynb")
    assert len(nb.cells) == 2
    assert nb.cells[1].source == "value = 3"